# Test — Cell Lattice + PINNs


In [ ]:
import numpy as np
from dyn_modelling.models.cell_lattice import build_lattice, simulate_cell_lattice, compute_distance_matrix, compute_weights, compute_rhs, external_signal
from dyn_modelling.methods.pinns import ANN, init_params, make_pinn, train
from dyn_modelling.utils.plotting import plot_graph, plot_trajectories, plot_loss, plot_parameter_evolution, plot_predictions
from dyn_modelling.utils.data import add_noise
import jax
import jax.numpy as jnp

ModuleNotFoundError: No module named 'dyn_modelling'

## 1. Build the lattice


In [ ]:
rows, cols = 5, 6
N = rows * cols

g = build_lattice(rows, cols)
print(f'Graph: {g.vcount()} nodes, {g.ecount()} edges')
plot_graph(g)

## 2. Simulate cell lattice


In [ ]:
# Parameters
a_params = np.array([2.4, 3.5, 2.0, 1.0])
l_params = np.array([1.0, 1.0, 0.2])
t_on, t_off = 80.0, 500.0
t_end, n_steps = 500.0, 100
neigh_order = 2

# Build RHS
dist_matrix = compute_distance_matrix(g)
rhs = compute_rhs(g, dist_matrix, a_params, l_params, neigh_order, t_on, t_off)

# Initial conditions
np.random.seed(1)
x0 = np.random.rand(N * 3)
t_eval = np.linspace(0, t_end, n_steps)

# Simulate
y = simulate_cell_lattice(rhs, x0, (0, t_end), t_eval)
print(f'Simulation done — y.shape: {y.shape}')

## 3. Plot trajectories — clean data


In [ ]:
plot_trajectories(t_eval, y)

## 4. Plot trajectories — noisy data


In [ ]:
y_noisy = add_noise(y, noise_level=0.1, seed=42)
plot_trajectories(t_eval, y_noisy)

## 5. Prepare data for PINN


In [ ]:
# PINN expects T shape (n_steps, 1) and Y shape (n_steps, 3*N)
T = t_eval[:, None]       # (100, 1)
Y = y.T                   # (100, 3*N) clean
Y_noisy = y_noisy.T       # (100, 3*N) noisy

## 6. Train PINN


In [ ]:
# Build jit-compiled loss and grad
batch_loss, grad_fn = make_pinn(g, N)

# Initialize ANN and model parameters
topology = [1, 40, 40, 40, 40, 3 * N]
ann_params = init_params(topology)
a_init = np.ones(4)
a_true = a_params.copy()

# Train
ann_params, a_list, loss_history = train(
    ann_params, a_init, T, Y_noisy,
    batch_loss, grad_fn,
    lmbd=0.5, n_epochs=5_000,
    lr_ann=1e-3, lr_model=1e-1, momentum=0.9
)

## 7. Plot loss


In [ ]:
plot_loss(loss_history)

## 8. Plot parameter evolution


In [ ]:
plot_parameter_evolution(a_list, a_true)

## 9. Plot predicted vs true trajectories


In [ ]:
batch_ANN = jax.vmap(lambda t: ANN(ann_params, t))
y_pred = np.array(batch_ANN(T))   # (n_steps, 3*N)

plot_predictions(t_eval, y_pred, Y_noisy)

## 10. Final identified parameters


In [ ]:
a_final = a_list[-1]
print(f'True parameters:      {a_true}')
print(f'Identified parameters: {a_final}')
print(f'Absolute error:        {np.abs(a_final - a_true)}')